In [ ]:
!pip install sentence-transformers
from google.colab import files
uploaded_files = files.upload() # upload the local files from the computer

In [ ]:
#import libraries
import pandas as pd
import numpy as np
import sys
from sentence_transformers import SentenceTransformer   # the SBERT model
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder

In [ ]:
sys.path.insert(0,'/content') # looking in /content for any local files we uploaded

In [ ]:
df = pd.read_csv('/content/data_preprocessed_csv.csv')

df['cleaned_abstract'] = df['cleaned_abstract'].fillna('') # handling missing values
df['abstract'] = df['abstract'].fillna('')

print("Dataset dimensions :",df.shape)
print("Columns in dataset :",df.columns.values)
print("\nFirst 2 rows of the abstract column :")
print(df['abstract'].head())

# Using MiniLM Model

In [ ]:
model_used = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# Extracts abstracts and encode them into numerical embeddings using the model
abstracts = df['abstract'].tolist()
embeddings = model_used.encode(abstracts,batch_size=64,convert_to_numpy=True,show_progress_bar=True)
print(f"Embeddings shape : {embeddings.shape}")

In [ ]:
np.save('/content/embeddings_nithya.npy',embeddings) # save the embeddings to the computer
files.download('/content/embeddings_nithya.npy')

In [ ]:
def manual_cossim(ppr1,ppr2):
    return np.dot(ppr1,ppr2)/(np.linalg.norm(ppr1)*np.linalg.norm(ppr2))  # Implementing cosine similarity manually like lab

# Picking 2 random papers and checking their similarity score
ppr1 = embeddings[0]
ppr2 = embeddings[1]
similarity_score = manual_cossim(ppr1,ppr2)
print("\nPaper 1 is about :",df['abstract'].iloc[0])
print("\nPaper 2 is about :",df['abstract'].iloc[1])
print(f"Their similarity score is : {similarity_score : }") # Range betwen 0 to 1, higher means more similar

In [ ]:
def table_of_similar_papers(query,top_k=10):
    query_embdng = model_used.encode([query],convert_to_numpy=True)[0]

    scores = []
    for e in embeddings:
        score = manual_cossim(query_embdng,e)
        scores.append(score)

    scores = np.array(scores)

    top_results = scores.argsort()[::-1][:top_k]

    results = df.iloc[top_results][['title','year','abstract']].copy()
    results['similarity score'] = scores[top_results].round(4)

    results['abstract'] = results['abstract'].str[:300]

    return results.reset_index(drop=True)

In [ ]:
table_of_similar_papers("deep learning")

In [ ]:
table_of_similar_papers("Natural language processing")

In [ ]:
number_of_clusters = 10
kmeans_clustering = KMeans(n_clusters=number_of_clusters,random_state=33)
df['kmeans_clusters'] = kmeans_clustering.fit_predict(embeddings)
print("Number of papers in each cluster : \n")
print(df['kmeans_clusters'].value_counts().sort_index())

In [ ]:
print("Few paper titles from each cluster :\n")
for c in range(number_of_clusters):
    paper_titles = df[df['kmeans_clusters'] == c]['title'].head(10)
    print(f"Few Papers in the Cluster {c} :")
    print(paper_titles.to_string(index=False))
    print()

In [ ]:
X_matrix = embeddings
mu = X_matrix.mean(axis=0)
Z = X_matrix - mu

cov_matrix = np.cov(Z,rowvar=False)

eigvals, eigvecs = np.linalg.eigh(cov_matrix)

idx = np.argsort(eigvals)[::-1]
eigvecs = eigvecs[:,idx]
pca_projection = Z @ eigvecs[:,:2]

df['pca_1'] = pca_projection[:,0]
df['pca_2'] = pca_projection[:,1]

fig, ax = plt.subplots(figsize=(15,7))

scatter = ax.scatter(df['pca_1'],df['pca_2'],c=df['kmeans_clusters'],cmap='tab10',s=2,alpha=0.5)

scatter = ax.scatter(df['pca_1'],df['pca_2'],c=df['kmeans_clusters'],cmap='tab10',s=2)

plt.colorbar(scatter,ax=ax,label='Clusters')

ax.set_title('PCA Clusters - Sbert Embeddings',fontsize=12)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
plt.tight_layout()

plt.savefig('/content/pca_clusters_nithya_sbert.png',dpi=150)
plt.show()

files.download('/content/pca_clusters_nithya_sbert.png')

In [ ]:
print(df.head())

In [ ]:
print(df['period'].value_counts().sort_index())

In [ ]:
print("Oldest Year :",df['year'].min())
print("Latest Year :",df['year'].max())
print("Total num of papers :",len(df))

# Experimenting with another model - Specter

In [ ]:
model_specter = SentenceTransformer('sentence-transformers/allenai-specter')  #using specter model to compare with MiniLM

In [ ]:
similarity_score_comparision = model_specter.encode([df['abstract'].iloc[0], df['abstract'].iloc[1]],convert_to_numpy=True)
specter_similarity_score = manual_cossim(similarity_score_comparision[0],similarity_score_comparision[1])
print(f"MiniLM similarity score : {similarity_score :}")
print(f"Specter similarity score : {specter_similarity_score :}")

In [ ]:
specter_model_embeddings = model_specter.encode(df['abstract'].tolist(),batch_size=64,convert_to_numpy=True,show_progress_bar=True)
print(specter_model_embeddings.shape)

In [ ]:
np.save('/content/specter_model_embeddings_nithya.npy',specter_model_embeddings)
#files.download('/content/specter_model_embeddings_nithya.npy')

In [ ]:
k_means_test = KMeans(n_clusters=10,random_state=33)

minilm_label = k_means_test.fit_predict(embeddings)
minilm_similarity_score = silhouette_score(embeddings,minilm_label,metric='cosine',sample_size=3000)

specter_label = k_means_test.fit_predict(specter_model_embeddings)
specter_similarity_score = silhouette_score(specter_model_embeddings,specter_label,metric='cosine',sample_size=3000)

print(f"MiniLM silhouette score : {minilm_similarity_score:}")
print(f"Specter silhouette score : {specter_similarity_score:}")

In [ ]:
# trying different k values

k_vals = []
for v in range(5,20):
    k_vals.append(v)

minilm_similarity_score_k = []
specter_similarity_score_k = []

for k in k_vals:
    k_means = KMeans(n_clusters=k,random_state=33)

    minilm_labels_k = k_means.fit_predict(embeddings)
    minilm_similarity_score_k.append(silhouette_score(embeddings,minilm_labels_k,metric='cosine',sample_size=3000))

    specter_labels_k = k_means.fit_predict(specter_model_embeddings)
    specter_similarity_score_k.append(silhouette_score(specter_model_embeddings,specter_labels_k,metric='cosine',sample_size=3000))

    print(f"k = {k}  MiniLM = {minilm_similarity_score_k[-1]:.4f}  Specter = {specter_similarity_score_k[-1]:.4f}")

In [ ]:
# plotting the silhouette scores

plt.figure(figsize=(10,6))
plt.plot(k_vals,minilm_similarity_score_k,marker='o',label='MiniLM')
plt.plot(k_vals,specter_similarity_score_k,marker='o',label='Specter')
plt.xlabel('K-vals')
plt.ylabel('Silhouette score')  # higher the better
plt.title('Silhouette Score Comparison - MiniLM VS Specter')
plt.legend()
plt.grid(True,alpha=0.3)
plt.savefig('/content/silhouette_comparison_minilm_vs_specter.png',dpi=150)
files.download('/content/silhouette_comparison_minilm_vs_specter.png')
plt.show()

# Using one more model MPNet for comparision


In [ ]:
model_mpnet = SentenceTransformer('all-mpnet-base-v2')  # experimenting with another model MPNet

In [ ]:
mpnet_model_embeddings = model_mpnet.encode(df['abstract'].tolist(),show_progress_bar=True)
print(mpnet_model_embeddings.shape)
np.save('/content/mpnet_model_embeddings_nithya.npy',mpnet_model_embeddings)
files.download('/content/mpnet_model_embeddings_nithya.npy')

In [ ]:
embeddings_dict = {'MiniLM':embeddings,'MPNet':mpnet_model_embeddings,'Specter':specter_model_embeddings} # storing all 3 model's embeddings as a dict for easier access
for model_name,embdngs in embeddings_dict.items():
    print(model_name,embdngs.shape) # to verify if all 3 gives same num of rows like it should

In [ ]:
print("Silhouette Scores for all 3 models :")  # comparing all 3 models
for model_name,embdngs in embeddings_dict.items():
    k_means = KMeans(n_clusters=10,random_state=33)
    labels = k_means.fit_predict(embdngs)
    silh_score = silhouette_score(embdngs,labels,metric='cosine',sample_size=3000)
    print(f"{model_name} : {silh_score:.4f}")

# Can the models tell the time periods apart?

In [ ]:
period_nums = LabelEncoder().fit_transform(df['period'])
for model_name,embdngs in embeddings_dict.items():
    silh_score = silhouette_score(embdngs,period_nums,metric='cosine',sample_size=9000)
    print(f"{model_name} : {silh_score:.4f}")

# PCA

In [ ]:
X_spec = specter_model_embeddings
mu_spec = X_spec.mean(axis=0)
Z_spec = X_spec - mu_spec

R_spec = np.cov(Z_spec,rowvar=False)
evals_spec, evecs_spec = np.linalg.eigh(R_spec)

idx_spec = np.argsort(evals_spec)[::-1]
evecs_spec = evecs_spec[:,idx_spec]

pca_coords = Z_spec @ evecs_spec[:,:2]

df['pca_x'] = pca_coords[:,0]
df['pca_y'] = pca_coords[:,1]

total_var = np.sum(evals_spec)
print(f"PC 1 : {(evals_spec[idx_spec[0]]/total_var)*100:.2f}%")
print(f"PC 2 : {(evals_spec[idx_spec[1]]/total_var)*100:.2f}%")

In [ ]:
fig,ax = plt.subplots(figsize=(10,7))

period_order = ['2005-2009','2010-2014','2015-2019','2020-2025']

for p in period_order:
    mask = df['period'] == p
    ax.scatter(df.loc[mask,'pca_x'],df.loc[mask,'pca_y'],label=p,s=2,alpha=0.4)

ax.set_title('PCA of Specter embeddings by time period')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.legend(markerscale=6)
plt.tight_layout()
plt.savefig('/content/pca_by_period_nithya.png',dpi=150)
files.download('/content/pca_by_period_nithya.png')
plt.show()

# Visualising the Trajectory of the field

In [ ]:
centroid_x_coord = []
centroid_y_coord = []

for p in period_order:
    mask = df['period'] == p

    mean_x = df.loc[mask,'pca_x'].mean()
    mean_y = df.loc[mask,'pca_y'].mean()

    centroid_x_coord.append(mean_x)
    centroid_y_coord.append(mean_y)
    print(f"Centroid for {p} : ({mean_x:.4f},{mean_y:.4f})")

In [ ]:
plt.figure(figsize=(12,8))

plt.scatter(df['pca_x'],df['pca_y'],s=1,alpha=0.1,color='gray')

plt.plot(centroid_x_coord,centroid_y_coord,'k--',alpha=0.5,linewidth=1) # dashed line connecting centroids to show change

for i, p in enumerate(period_order):
    plt.scatter(centroid_x_coord[i],centroid_y_coord[i],s=150,zorder=5)
    plt.text(centroid_x_coord[i]+0.1,centroid_y_coord[i]+0.1,p,fontsize=10,fontweight='bold',color='black')

plt.title('Trajectory of Research Topics')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.grid(True,linestyle=':',alpha=0.6) # Standard worksheet grid style
plt.legend(loc='upper right')
plt.show()
plt.savefig('/content/trajectory_topics_nithya.png',dpi=150)
files.download('/content/trajectory_topics_nithya.png')